In [1]:

from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Trained adapter path
peft_model_id = "/home/jovyan/Works/MSc_Thesis/mh_function_calling/model_training/finetuned_models/llama3.2_3b_mh_function_calling_v2" 

print("Loading model...")

# Load PEFT config to get base model info
config = PeftConfig.from_pretrained(peft_model_id)
print(f"Base model: {config.base_model_name_or_path}")

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map={"": 0},
    torch_dtype=torch.bfloat16
)

# Load tokenizer from the adapter (contains special tokens)
tokenizer = AutoTokenizer.from_pretrained(peft_model_id)
print(f"Tokenizer vocab size: {len(tokenizer)}")

# Resize model to accommodate new tokens
model.resize_token_embeddings(len(tokenizer))

# Load the adapter weights
model = PeftModel.from_pretrained(model, peft_model_id)
model.to(torch.bfloat16)
model.eval()

print("Model loaded successfully!")

Loading model...
Base model: meta-llama/Llama-3.2-3B-Instruct


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.95it/s]
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Tokenizer vocab size: 128260


/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:574: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['model.embed_tokens', 'lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(


Model loaded successfully!


In [21]:
# Test the model with the CORRECT tokenizer (the one with special tokens)
def test_model(patient_id, query, max_length=1024):
    # Format exactly like training data - include the explicit instruction
    messages = [
        {
            "role": "system",
            "content": "You are a helpful mental health assistant. When users ask for health data, use the available functions to retrieve the information. Think through your response before choosing a function. Use <think>your reasoning</think> tags to show your thought process, then use <function>json</function> tags to call the appropriate function."
        },
        {
            "role": "user",
            "content": f"Patient ID: {patient_id}\nQuery: {query}"
        }
    ]
    
    # Apply chat template using the LOADED tokenizer
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # Tokenize and generate
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs['input_ids'].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            temperature=0.01,              
            top_p=0.95,                 
            repetition_penalty=1.0,     
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            
            
        )
    
    response_tokens = outputs[0][input_length:]
    response = tokenizer.decode(response_tokens, skip_special_tokens=False)
    
    # Clean up the eos for clear response
    response = response.replace(tokenizer.eos_token, "").strip()
    
    return response

# Test with the loaded model and tokenizer
test_query = "Could you please pull my sleep and heart rate data?"
response = test_model("UID_123", test_query)
print(f"Response: {response}")
print("="*80)

Response: <think>
The user is asking about their sleep and heart rate - no specific timeframe given. For sleep data, I need to call get_sleep_data. For heart rate data, I need to call get_heart_rate_data. Since they mentioned 'past few days', I'll default to 3 days to show recent patterns without overwhelming them with data.
</think>

<function>
{
  "name": "get_sleep_data",
  "parameters": {
    "patient_id": "UID_123",
    "num_days": 3
  }
}
</think>
